# The goal is to collect all the data form the different years and compare the best algos from each year to determine the best of the best : BBOB- noisy test Suite

# 1) Collecting the data from the different years and making one single file with all the best algos from all the different years.

We first build a "table" where we will agregate all the data from all of the different years.

In [1]:
import pandas as pd
from pathlib import Path

# === 1. Load all yearly CSVs from the "results" folder ===
folder = Path("results")
all_files = sorted(folder.glob("bbob-noisy_*.csv"))

dfs = []
for f in all_files:
    year = int(f.stem.split("_")[-1])
    df = pd.read_csv(f)
    df["year"] = year
    dfs.append(df)

# Merge everything into a single DataFrame
df_all = pd.concat(dfs, ignore_index=True)

# === 2. Find, for each (dim, func, target), the entry with smallest ERT ===
idx = df_all.groupby(["dimension", "function_id", "target"])["best_ERT"].idxmin()
df_best_overall = df_all.loc[idx, ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]]

# Sort nicely
df_best_overall = df_best_overall.sort_values(["dimension", "function_id", "target"]).reset_index(drop=True)

# === 3. Display result ===
print(" Global best algorithm across all years (smallest ERT for each dim/function/target):\n")
print(df_best_overall.head(20))

# Optionally save to file
df_best_overall.to_csv("results/global_best_algos_bbob-noisy.csv", index=False)


 Global best algorithm across all years (smallest ERT for each dim/function/target):

    dimension  function_id  year         best_algorithm        target  \
0           2          101  2009   FULLNEWUOA_ros_noisy  1.000000e-08   
1           2          101  2009   FULLNEWUOA_ros_noisy  1.000000e-05   
2           2          101  2009    SNOBFIT_huyer_noisy  1.000000e-03   
3           2          101  2009    SNOBFIT_huyer_noisy  1.000000e-02   
4           2          101  2009    SNOBFIT_huyer_noisy  1.000000e-01   
5           2          102  2009   FULLNEWUOA_ros_noisy  1.000000e-08   
6           2          102  2009   FULLNEWUOA_ros_noisy  1.000000e-05   
7           2          102  2009   FULLNEWUOA_ros_noisy  1.000000e-03   
8           2          102  2009    SNOBFIT_huyer_noisy  1.000000e-02   
9           2          102  2009    SNOBFIT_huyer_noisy  1.000000e-01   
10          2          103  2009   FULLNEWUOA_ros_noisy  1.000000e-08   
11          2          103  2009    SN

### General table summing up nice stats.

- Where the best algos are located (year)
- Which algorithms are the dominant winners (counter)


In [9]:
# Count wins per year
print(df_best_overall["year"].value_counts())

# Count wins per algorithm
print(df_best_overall["best_algorithm"].value_counts().head(10))


2010    367
2009    223
2012    220
2016     90
Name: year, dtype: int64
IPOPsaACM_loshchilov_noisy           176
IPOP-ACTCMA-ES_ros_noisy             133
1komma4mirser_brockhoff_noisy        117
IPOP-CMA-ES_ros_noisy                 85
SNES_schaul_noisy                     37
PSAaSmC-CMA-ES_Nishida_bbob-noisy     36
FULLNEWUOA_ros_noisy                  35
BIPOP-CMA-ES_hansen_noisy             28
SNOBFIT_huyer_noisy                   27
PSAaLmC-CMA-ES_Nishida_bbob-noisy     22
Name: best_algorithm, dtype: int64


The table bellow repertoriates the dimension for which the algorithms are best performing. 

In [10]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===

# Count how many times each algorithm appears as best within each dimension
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()  # count occurrences
    .reset_index(name="count")
)

# Sort within each dimension by count descending
algo_counts = (
    algo_counts
    .sort_values(["dimension", "count"], ascending=[True, False])
)

# For convenience: for each dimension, keep rank of algorithms by count
algo_counts["rank"] = algo_counts.groupby("dimension")["count"].rank(method="first", ascending=False)


pivot_table = algo_counts.pivot(index="best_algorithm", columns="dimension", values="count").fillna(0).astype(int)
pivot_table["Total"] = pivot_table.sum(axis=1)
pivot_table = pivot_table.sort_values("Total", ascending=False)

print("\n Overall frequency of being best by algorithm and dimension (top 10):\n")
print(pivot_table.head(10))



 Overall frequency of being best by algorithm and dimension (top 10):

dimension                           2   3   5  10  20  40  Total
best_algorithm                                                  
IPOPsaACM_loshchilov_noisy         12  29  42  38  55   0    176
IPOP-ACTCMA-ES_ros_noisy           20  22  14  13  11  53    133
1komma4mirser_brockhoff_noisy      11  17  18  18  16  37    117
IPOP-CMA-ES_ros_noisy              10   5  12  13  20  25     85
SNES_schaul_noisy                   4   8   9   9   3   4     37
PSAaSmC-CMA-ES_Nishida_bbob-noisy   2   8   4   8  14   0     36
FULLNEWUOA_ros_noisy               12   7   7   8   1   0     35
BIPOP-CMA-ES_hansen_noisy           4   3   6   9   4   2     28
SNOBFIT_huyer_noisy                 9   9   7   2   0   0     27
PSAaLmD-CMA-ES_Nishida_bbob-noisy   3   9  10   0   0   0     22


# 3) Results : 2 different types of tables. 
The goal is to nicely present our results in tables summing up which are the best algorithms over all. 
The definition of “best algorithm” is NOT different between the tables. What is different is the DATA they aggregate.


- We are going to present our results in 2 different ways. A first one being getting the best algo over all dimensions, function AND aggregating over every target precision. 

- A second way is to determine the best algo over all dimensions, functions BUT only for a specific target, and aggregate only on that target. 

## 3.1) Aggregating over every target 



### 3.1.1) Table of the best algos
Here we are building a table that sums up the best algorithms for each dimension: there are 6 different dimensions. We can also see the count for how many times that algorithm was the best in that specific dimension 

The columns of this table are the different dimmension, and the lines of the table represent the ranking of the best algorithms. In the first line we will have the best algorithm for each dimension. The second line will represnt the 2nd best algorithms for each dimension etc... 

In the sence that we are counting wins across all targets at once. Then rank algorithms per dimension across this big mixture.
i.e. “Across ALL target precisions, which algorithm wins the most often in each dimension?”

In [11]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()
    .reset_index(name="count")
)

# Sort within each dimension by how often each algo was best
algo_counts = algo_counts.sort_values(["dimension", "count"], ascending=[True, False])

# Add ranking per dimension
algo_counts["rank"] = (
    algo_counts
    .groupby("dimension")["count"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# === NEW PART: build a tuple column (algo, count) ===
algo_counts["algo_tuple"] = list(zip(algo_counts["best_algorithm"], algo_counts["count"]))

# === 5. Pivot: rows = rank, columns = dimension, values = (algo_name, count) tuple ===
algo_ranking_table = algo_counts.pivot(index="rank", columns="dimension", values="algo_tuple")

# Sort dimensions in ascending order
algo_ranking_table = algo_ranking_table.reindex(sorted(algo_ranking_table.columns), axis=1)

# === 6. Display neatly ===
print("\n Ranking of best algorithms per dimension (with counts):\n")
from IPython.display import display
display(algo_ranking_table.head(10))  # show top 10 ranks



 Ranking of best algorithms per dimension (with counts):



dimension,2,3,5,10,20,40
rank,,,,,,
1,"(IPOP-ACTCMA-ES_ros_noisy, 20)","(IPOPsaACM_loshchilov_noisy, 29)","(IPOPsaACM_loshchilov_noisy, 42)","(IPOPsaACM_loshchilov_noisy, 38)","(IPOPsaACM_loshchilov_noisy, 55)","(IPOP-ACTCMA-ES_ros_noisy, 53)"
2,"(FULLNEWUOA_ros_noisy, 12)","(IPOP-ACTCMA-ES_ros_noisy, 22)","(1komma4mirser_brockhoff_noisy, 18)","(1komma4mirser_brockhoff_noisy, 18)","(IPOP-CMA-ES_ros_noisy, 20)","(1komma4mirser_brockhoff_noisy, 37)"
3,"(IPOPsaACM_loshchilov_noisy, 12)","(1komma4mirser_brockhoff_noisy, 17)","(IPOP-ACTCMA-ES_ros_noisy, 14)","(IPOP-ACTCMA-ES_ros_noisy, 13)","(1komma4mirser_brockhoff_noisy, 16)","(IPOP-CMA-ES_ros_noisy, 25)"
4,"(1komma4mirser_brockhoff_noisy, 11)","(PSAaLmD-CMA-ES_Nishida_bbob-noisy, 9)","(IPOP-CMA-ES_ros_noisy, 12)","(IPOP-CMA-ES_ros_noisy, 13)","(PSAaSmC-CMA-ES_Nishida_bbob-noisy, 14)","(CMAEGS_finck_noisy, 5)"
5,"(AMALGAM_bosman_noisy, 11)","(SNOBFIT_huyer_noisy, 9)","(PSAaLmD-CMA-ES_Nishida_bbob-noisy, 10)","(BIPOP-CMA-ES_hansen_noisy, 9)","(IPOP-ACTCMA-ES_ros_noisy, 11)","(SNES_schaul_noisy, 4)"
6,"(IPOP-CMA-ES_ros_noisy, 10)","(PSAaLmC-CMA-ES_Nishida_bbob-noisy, 8)","(SNES_schaul_noisy, 9)","(SNES_schaul_noisy, 9)","(RCGA_tran_noisy, 5)","(BIPOP-CMA-ES_hansen_noisy, 2)"
7,"(GLOBAL_pal_noisy, 9)","(PSAaSmC-CMA-ES_Nishida_bbob-noisy, 8)","(FULLNEWUOA_ros_noisy, 7)","(FULLNEWUOA_ros_noisy, 8)","(BIPOP-CMA-ES_hansen_noisy, 4)","(CMA-ESPLUSSEL_auger_noisy, 2)"
8,"(SNOBFIT_huyer_noisy, 9)","(SNES_schaul_noisy, 8)","(SNOBFIT_huyer_noisy, 7)","(PSAaSmC-CMA-ES_Nishida_bbob-noisy, 8)","(MCS_huyer_noisy, 3)","(IPOP-SEP-CMA-ES_ros_noisy, 2)"
9,"(MCS_huyer_noisy, 7)","(FULLNEWUOA_ros_noisy, 7)","(BIPOP-CMA-ES_hansen_noisy, 6)","(PSAaSmD-CMA-ES_Nishida_bbob-noisy, 8)","(SNES_schaul_noisy, 3)","(RCGA_tran_noisy, 2)"


We have noticed that the best algorithms come as batches of 6 algorithms per line. So hence when we are asking for the N best algorithms, where N<6, it is complicated to choose which algorithms to return, also some algos may be the best in different dimensions. 


For example let's say we are asking for the 8 best algorithms. In this case we will go through the following lines of the table. There are several cases we need to consider to choose these 8 best algorithms.

- One scenario, is that in the following line, there are the names of algorithms that were already mentioned in the first line. In this case we don't add their name again. 
- The second scenario is we are in fact not able to return the names of 8 algorithms, because all the algorithms have performed equally for their respective dimension. Instead of choosing only 8, we will return the names of all the algorithms that are mentioned and are on the same line. We also notify that we are not returning 8 algorithms, but a bit more, here 10.
- A further step would be to really restrict the number to 8. This means we need to find a way to classify the algorithms that are on the same line. An easy way is to select the one that has the highest occurence of "best" algorithm. 

### 3.2.1) List of the best algos according to the table

Here our goal is to return N number of best algo depensing on how many best algo the user is asking for. We will treat every algo on a same row equally, not one being better than the other. 

In [15]:
def get_top_algorithms(algo_ranking_table, N_min):
    """
    Collect algorithms from top ranks (rows) across all dimensions
    until at least N_min unique algorithms are found.
    """
    seen_algos = set()
    row_idx = 0

    # Keep adding rows until we have at least N_min unique algorithms
    while len(seen_algos) < N_min and row_idx < len(algo_ranking_table):
        row_algos = algo_ranking_table.iloc[row_idx].dropna().unique()

        # Extract only the algorithm names (first element of tuple)
        algo_names = {algo_tuple[0] for algo_tuple in row_algos}

        # Add these clean names to the seen set
        seen_algos.update(algo_names)
        row_idx += 1

    algo_list = sorted(seen_algos)

    print(f"\n Requested {N_min} best algorithms.")
    print(f" Returning {len(algo_list)} unique algorithms (reached rank {row_idx}).\n")
    print(" List of selected algorithms:\n")
    for algo in algo_list:
        print(f" - {algo}")

    return algo_list


# === Interactive part ===
try:
    # Ask user for number of desired algorithms
    N_input = int(input("How many best algorithms do you want? "))
    if N_input < 1:
        print(" Minimum number of best algorithms is 1. Using N=1.")
        N_input = 1

    # Compute and display
    best_algos = get_top_algorithms(algo_ranking_table, N_min=N_input)

except ValueError:
    print("Invalid input. Please enter an integer number (e.g., 6 or 7).")


How many best algorithms do you want? 1

 Requested 1 best algorithms.
 Returning 2 unique algorithms (reached rank 1).

 List of selected algorithms:

 - IPOP-ACTCMA-ES_ros_noisy
 - IPOPsaACM_loshchilov_noisy


## 3.2) Choice of a specific target precision for the bets algo.

### 3.2.1) Table dpending on a specific target precision


- We filter by ONE target precision of our choice.

- Count wins only for that target.

- Rank algos per dimension based only on those wins. 

i.e. “For target = X, which algorithm wins the most functions in each dimension?”

In [6]:
import pandas as pd

# --- assuming df_best_overall is already built as before ---
# columns: ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]


def make_dimension_ranking_table_with_counts(df_best_overall, target):
    """
    For a given target:
    - Count, for each dimension, how many times each algorithm was best.
    - Rank algorithms within each dimension by this count.
    - Pivot into a table: rows = rank, columns = dimension,
      values = (algorithm, count) tuples.
    """
    # Filter by target
    df_t = df_best_overall[df_best_overall["target"] == target].copy()
    if df_t.empty:
        raise ValueError(f"No data found for target = {target}")

    # Aggregate: count how many times each algorithm is best in each dimension
    algo_counts = (
        df_t.groupby(["dimension", "best_algorithm"])
        .size()
        .reset_index(name="count")
    )

    # Sort inside each dimension
    algo_counts = algo_counts.sort_values(
        ["dimension", "count"], ascending=[True, False]
    )

    # Rank per dimension
    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # Build tuple column
    algo_counts["algo_tuple"] = list(
        zip(algo_counts["best_algorithm"], algo_counts["count"])
    )

    # Pivot: rows = rank, columns = dimension, values = tuple
    ranking_table = algo_counts.pivot(
        index="rank", columns="dimension", values="algo_tuple"
    )

    # Sort dimensions
    ranking_table = ranking_table.reindex(
        sorted(ranking_table.columns), axis=1
    )

    return ranking_table


# === Interactive part: only ask for target ===
try:
    print("Available targets:", df_best_overall["target"].unique())

    # User input
    target_input = float(input("\nSelect a target precision (e.g., 1e-8 or 1e-5): "))

    # Build the table
    algo_ranking_table = make_dimension_ranking_table_with_counts(
        df_best_overall,
        target_input
    )

    print(f"\nRanking table built for target = {target_input}")
    from IPython.display import display
    display(algo_ranking_table.head(3))

except ValueError:
    print("Invalid target input. Please enter a numeric target.")


Available targets: [1.e-08 1.e-05 1.e-03 1.e-02 1.e-01]

Select a target precision (e.g., 1e-8 or 1e-5): 1e-8

Ranking table built for target = 1e-08


dimension,2,3,5,10,20,40
rank,,,,,,
1,"(IPOP-ACTCMA-ES_ros_noisy, 5)","(IPOPsaACM_loshchilov_noisy, 6)","(IPOPsaACM_loshchilov_noisy, 8)","(IPOPsaACM_loshchilov_noisy, 7)","(IPOPsaACM_loshchilov_noisy, 10)","(IPOP-ACTCMA-ES_ros_noisy, 13)"
2,"(FULLNEWUOA_ros_noisy, 4)","(IPOP-ACTCMA-ES_ros_noisy, 5)","(1komma4mirser_brockhoff_noisy, 5)","(1komma4mirser_brockhoff_noisy, 4)","(IPOP-ACTCMA-ES_ros_noisy, 6)","(1komma4mirser_brockhoff_noisy, 7)"
3,"(1komma4mirser_brockhoff_noisy, 3)","(1komma4mirser_brockhoff_noisy, 4)","(IPOP-ACTCMA-ES_ros_noisy, 3)","(IPOP-ACTCMA-ES_ros_noisy, 4)","(1komma4mirser_brockhoff_noisy, 3)","(IPOP-CMA-ES_ros_noisy, 2)"


# 4) plots 

In [12]:
import cocopp
cocopp.main(['IPOP-ACTCMA-ES_ros_noisy'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\IPOP-ACTCMA-ES_ros_noisy_111923h2855
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob-noisy.tar.gz ...
    archive extracted to folder C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs\.extracted_best2009-bbob-noisy ...
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob-noisy.tar.gz
  done (Wed Nov 19 23:29:18 2025).
{message : DeprecationWarning('The py23 module has been deprecated and will be removed in a future release. Please update your code.'), category : 'DeprecationWarning', filename : 'C:\\Users\\elsaf\\anaconda3\\Lib

C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pplogloss.py:776: RuntimeWarning: divide by zero encountered in log10
  ydata.append(np.log10(list(data[f][i] for f in data)))
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pplogloss.py:776: RuntimeWarning: divide by zero encountered in log10
  ydata.append(np.log10(list(data[f][i] for f in data)))


  done (Wed Nov 19 23:31:21 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\IPOP-ACTCMA-ES_ros_noisy_111923h2855
Setting changes in `cocopp.genericsettings` compared to default:
    simulated_runlength_bootstrap_sample_size: from 30 to 10.098990100989901
    foreground_algorithm_list: from [] to ['C:\\Users\\elsaf\\Ap...
ALL done (Wed Nov 19 23:31:22 2025).


DictAlg([(('IPOP-ACTCMA-ES_ros_noisy', ''),
          [DataSet(IPOP-ACTCMA-ES_ros_noisy on f101 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f102 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f103 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f104 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f105 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f106 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f107 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f108 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f109 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f110 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f111 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f112 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f113 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f114 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f115 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f116 2-D),
           DataSet(IPOP-ACTC

Asboslute best results 

In [8]:
import cocopp
cocopp.main(['IPOP-ACTCMA-ES_ros_noisy','IPOPsaACM_loshchilov_noisy']) 

Post-processing (2+)
  Using 2 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
  Will generate output data in folder ppdata\IPOP-_IPOPs_112700h4113
    this might take several minutes.
ECDF runlength ratio graphs...
  done (Thu Nov 27 00:41:20 2025).
ECDF runlength graphs...
  done (Thu Nov 27 00:41:25 2025).
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2009-bbob-noisy.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob-noisy.tar.gz
  done (Thu Nov 27 00:41:26 2025).
  done (Thu No

DictAlg([(('IPOP-ACTCMA-ES_ros_noisy', ''),
          [DataSet(IPOP-ACTCMA-ES_ros_noisy on f101 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f102 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f103 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f104 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f105 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f106 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f107 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f108 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f109 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f110 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f111 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f112 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f113 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f114 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f115 2-D),
           DataSet(IPOP-ACTCMA-ES_ros_noisy on f116 2-D),
           DataSet(IPOP-ACTC